# Life2Lang — Pretraining

Pretrain a T5 encoder-decoder on protein sequences using **span corruption**.

**Outputs**
- Experiment tracked on [Weights & Biases](https://wandb.ai)
- Final model pushed to 🤗 Hub as `khairi/life2lang-base-pt`

**Runtime**: GPU required (A100 / V100 recommended for base-size model).

## 1 · Install

In [ ]:
!pip install -q git+https://github.com/abidikhairi/life2lang.git
!pip install -q wandb millify

## 2 · Authenticate

In [ ]:
import wandb
wandb.login()

In [ ]:
from huggingface_hub import login
login()  # paste a token with write access to khairi/life2lang-base-pt

## 3 · Configuration

In [ ]:
# ── Data ──────────────────────────────────────────────────────────────────
DATASET_ID = "khairi/life2lang-pretraining-dataset"

# ── Model ─────────────────────────────────────────────────────────────────
BASE_MODEL   = "khairi/life2lang-base"
HUB_MODEL_ID = "khairi/life2lang-base-pt"
OUTPUT_DIR   = "/tmp/life2lang-base-pt"

# ── Training ──────────────────────────────────────────────────────────────
BATCH_SIZE           = 4
EVAL_BATCH_SIZE      = 8
GRADIENT_ACCUM_STEPS = 8    # effective batch = 32
NUM_EPOCHS           = 5
LEARNING_RATE        = 1.2e-3
WEIGHT_DECAY         = 0.01
MAX_GRAD_NORM        = 0.1
WARMUP_RATIO         = 0.15
LR_SCHEDULER         = "cosine"
EVAL_STEPS           = 500
SAVE_STEPS           = 500
LOGGING_STEPS        = 100

# ── W&B ───────────────────────────────────────────────────────────────────
WANDB_PROJECT  = "life2lang"
WANDB_RUN_NAME = "pretraining-base"

## 4 · Environment check

In [ ]:
import os
import torch
from millify import millify

from life2lang.utils import (
    print_gpu_info,
    print_gpu_memory,
    clean_gpu_memory,
    count_trainable_parameters,
)

print_gpu_info()
print_gpu_memory()

os.environ["WANDB_PROJECT"] = WANDB_PROJECT

## 5 · Load dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset(DATASET_ID)
train_data = dataset["train"]
valid_data = dataset["validation"]

print(f"Train size : {len(train_data):,}")
print(f"Valid size : {len(valid_data):,}")
print(f"Columns    : {train_data.column_names}")
train_data[0]

## 6 · Tokenise

In [ ]:
from life2lang.models import T5Tokenizer
from life2lang.utils import tokenize_examples

tokenizer = T5Tokenizer.from_pretrained(BASE_MODEL)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Vocab size : {tokenizer.vocab_size}")

_tokenize = lambda x: tokenize_examples(x, tokenizer)
keep_cols  = ["input_ids", "attention_mask", "labels"]

train_data = train_data.map(_tokenize, batched=True, batch_size=64).select_columns(keep_cols)
valid_data = valid_data.map(_tokenize, batched=True, batch_size=64).select_columns(keep_cols)

print(f"Sample input length : {len(train_data[0]['input_ids'])}")
print(f"Sample label length : {len(train_data[0]['labels'])}")

## 7 · Model

In [ ]:
from life2lang.models import T5ForConditionalGeneration

model = T5ForConditionalGeneration.from_pretrained(BASE_MODEL)

n_params = count_trainable_parameters(model)
print(f"Trainable parameters : {millify(n_params)} ({n_params:,})")
print(f"Max context distance : {model.config.relative_attention_max_distance}")

## 8 · Train

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    # Batching
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUM_STEPS,

    # Schedule
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    optim="adamw_torch",
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=MAX_GRAD_NORM,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type=LR_SCHEDULER,

    # Evaluation & checkpointing
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_steps=SAVE_STEPS,
    logging_steps=LOGGING_STEPS,
    save_total_limit=2,
    load_best_model_at_end=True,

    # Logging
    report_to="wandb",
    run_name=WANDB_RUN_NAME,

    # Hub
    push_to_hub=True,
    hub_model_id=HUB_MODEL_ID,
    hub_strategy="checkpoint",

    remove_unused_columns=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=valid_data,
    data_collator=data_collator,
)

clean_gpu_memory()
trainer.train()

## 9 · Save & push

In [ ]:
final_dir = OUTPUT_DIR + "/final_model"

trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)
print(f"Saved locally → {final_dir}")

trainer.push_to_hub(commit_message="final pretrained model")
tokenizer.push_to_hub(HUB_MODEL_ID)
print(f"Pushed → https://huggingface.co/{HUB_MODEL_ID}")

In [ ]:
wandb.finish()